# No stimulation Demo Experiment
This notebook showcases a demo timelapse experiment, where no stimulation is applied to the cells. The experiment runs as a timelapse, and the cells are imaged at regular intervals. In this case the feature extractor is set to the baseline feature extractors, which just extract label position and area. For tracking trackpy is used and stardist is used for segmentation. 

As this is a demo experiment, it runs on the demo hardware provided by micro-manager. The demo hardware is a simulated hardware that mimics the behavior of a real microscope.

**Note:** This notebook has been updated to use the new API pattern from `04_full_FOV_stimulation_ERK_new_API.ipynb`. The acquisition schedule is now generated using `utils.generate_df_acquire` for better clarity and reproducibility.

### Import required libraries

In [20]:
import os
import time
from rtm_pymmcore.core.data_structures import Fov, Channel, StimTreatment
from pprint import pprint
import pandas as pd
import numpy as np
import dataclasses
import random

### Experimental Settings

In [ ]:
from rtm_pymmcore.microscope.demo import MMDemo

# Path to old data for simulation (demo purposes only)
path_with_old_data_for_simulation = os.path.join(
    ".", "test_exp_data", "00_01_demo_imgs"
)

mic = MMDemo()

In [ ]:
## Configuration options - set experiment timing, storage and pipeline parameters
# General timing and frame counts:
N_FRAMES = 2  # number of timesteps

# If you want the notebook/script to wait before starting the experiment, set this (hours).
SLEEP_BEFORE_EXPERIMENT_START_in_H = 0

# Timing for acquisition: interval between timesteps (seconds) and approximate time per FOV (seconds).
TIME_BETWEEN_TIMESTEPS = 2  # seconds between timesteps
TIME_PER_FOV = 1  # seconds per FOV (camera + stage moves + overhead)

## Storage path for the experiment - change to your desired directory and experiment name.
base_path = "C:\\Users\\Alex\\Ausbildung\\PhD_temp\\test_exp"
experiment_name = "exp_test"
path = os.path.join(base_path, experiment_name)


## Channels: images to acquire each timestep.
# Each Channel(...) maps to a microscope channel name configured in Micro-Manager/your device.
# If exposure or power is omitted, the hardware default (set in the device/GUI) will be used.
channels = []
channels.append(
    Channel(
        config="DAPI",
        exposure=150,
        group="Channel",
        power=2,
        device_name="LED",
        property_name="State",
    )
)
channels.append(Channel(config="Cy5", exposure=150))

# Experimental condition(s): a list of labels assigned to FOVs.
condition = ["FGFR_high"]  # Example of adding a condition to the dataframe
# condition = ["optoFGFR_high"] * 24 + ["optoFGFR"] * 24  # Example: multiple conditions

# If using wellplates, set how many FOVs per well. Set to None if not using wellplates.
n_fovs_per_well = None  ## number of FOVs per well; use None for free-FOV experiments

## Define the Tools that you are using for the experiment (segmentors, trackers, feature extractors)
from rtm_pymmcore.segmentation.remote import SegmentatorImagingServerKit
from rtm_pymmcore.tracking.trackpy import TrackerTrackpy
from rtm_pymmcore.feature_extraction.simple import SimpleFE

segmentators = [
    {
        "name": "labels",
        "class": SegmentatorImagingServerKit(
            "http://130.92.82.27:8000",
            "cellpose",
            min_size=50,
        ),
        "use_channel": 0,
        "save_tracked": True,
    },
]

stimulator = None  # No stimulation in this experiment
feature_extractor = SimpleFE("labels")
tracker = TrackerTrackpy()

from rtm_pymmcore.core.pipeline import ImageProcessingPipeline

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=feature_extractor,
    tracker=tracker,
    stimulator=stimulator,
)

### GUI - Napari Micromanager

#### Load GUI

In [5]:
### Base GUI ###
from napari_micromanager import MainWindow
import napari

viewer = napari.Viewer()
mm_wdg = MainWindow(viewer)
mm_wdg._mmc = mic.mmc
viewer.window.add_dock_widget(mm_wdg)
data_mda_fovs = None
load_from_file = False

### Map Experiment to FOVs

#### If FOVs already saved - Reload them from file

In [25]:
load_from_file = True

### Generate dataframe for acquisition using utils helpers

In [26]:
# Generate FOV objects from microscope / viewer (or load from file)
import rtm_pymmcore.core.utils as utils

if not load_from_file:
    # Generate FOVs from MDA widget or viewer
    fovs = utils.generate_fov_objects(mic, viewer=viewer)
else:
    # When loading from file, use generate_fov_objects with filename parameter
    fovs = utils.generate_fov_objects(mic, filename=os.path.join(path, "fovs.json"))

# Generate acquisition dataframe using utils helper (no stimulation)
df_acquire = utils.generate_df_acquire(
    fovs,
    n_frames=N_FRAMES,
    time_between_timesteps=TIME_BETWEEN_TIMESTEPS,
    time_per_fov=TIME_PER_FOV,
    channels=channels,
    condition=condition,
)

# Display settings for better readability
pd.set_option("display.max_columns", None)
pd.set_option("display.expand_frame_repr", True)
df_acquire

Total Experiment Time: 0.0005555555555555556h


,fov_object,fov,fov_x,fov_y,fov_z,fov_name,timestep,time,channels,fname,cell_line
0,<rtm_pymmcore.core.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,0,0,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",000_00000,FGFR_high
1,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,0,0,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",001_00000,FGFR_high
2,<rtm_pymmcore.core.data_structures.Fov object at 0x...,0,0.00,0.0,0.0,0,1,2,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",000_00001,FGFR_high
3,<rtm_pymmcore.core.data_structures.Fov object at 0x...,1,20.01,0.0,0.0,1,1,2,"({'name': 'DAPI', 'exposure': 150, 'group': 'C...",001_00001,FGFR_high


### Run experiment

In [ ]:
from rtm_pymmcore.core.controller import ControllerSimulated
from rtm_pymmcore.core.conversion import df_to_events

for _ in range(0, SLEEP_BEFORE_EXPERIMENT_START_in_H * 3600):
    time.sleep(1)

try:
    mm_wdg._core_link.cleanup()

except:
    pass

print(f"Running in simulated mode with old data from {path_with_old_data_for_simulation}")
events = df_to_events(df_acquire)
ctrl = ControllerSimulated(mic, pipeline, path_with_old_data_for_simulation)
ctrl.run_experiment(events)

mic.post_experiment()
time.sleep(10)

utils.generate_exp_data_from_tracks(path)

from napari_micromanager._core_link import CoreViewerLink

if "viewer" in locals():
    mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)

### Function to re-connect link with GUI if manually broken

The following functions can be used to manually re-make the connection between the GUI and the running rtm-pymmcore script. However, normally you don't need to execute them. 

In [ ]:
### Manually reconnect pymmcore with napari-micromanager
from napari_micromanager._core_link import CoreViewerLink

mm_wdg._core_link = CoreViewerLink(viewer, mic.mmc)